# HotpotQA HDBSCAN Validation

This notebook validates `hdbscan_cluster()` on the HotpotQA embedding store.

It provides:

1. A full-span clustering pass over every HotpotQA span.
2. Noise reassignment by cosine similarity against fixed cluster mean embeddings.
3. Exhaustive label assignment for every embedding in multi-cluster spans using the same majority-vote idea as `merge_duplicate_description_proto_nodes()`.


In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")


Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
from collections import Counter, defaultdict
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import spacy
import torch
from IPython.display import display
from sentence_transformers import CrossEncoder
from spacy.tokenizer import Tokenizer
from spacy.util import compile_infix_regex
from transformers import AutoModel, AutoTokenizer

from text_processing import (
    clean_text,
    encode_chunk_batch,
    extract_important_spans,
    get_token_indices_for_phrase,
    normalize_text,
)
from utils import (
    build_hotpot_retrieval_dataset,
    build_wikidata_candidate_bank,
    extract_cross_encoder_scores,
    hdbscan_cluster,
    load_wikidata_definition_candidates,
)

pd.set_option("display.max_colwidth", None)


In [3]:
HOTPOT_FILE_CANDIDATES = [
    REPO_ROOT / "jupyter_notebooks" / "hotpot_dev_distractor_v1.json",
    REPO_ROOT / "hotpot_dev_distractor_v1.json",
]
HOTPOT_EMBEDDING_STORE_CANDIDATES = [
    REPO_ROOT / "jupyter_notebooks" / "hotpot_phrase_token_embedding_index_all.pkl",
    REPO_ROOT / "hotpot_phrase_token_embedding_index_all.pkl",
]
DEFAULT_CLUSTER_RESULTS_PATH = REPO_ROOT / "jupyter_notebooks" / "hotpot_hdbscan_cluster_results.pkl"
DEFAULT_LABEL_RESULTS_PATH = REPO_ROOT / "jupyter_notebooks" / "hotpot_hdbscan_label_results.pkl"
DEFAULT_MODEL_NAME = "cross-encoder/nli-deberta-v3-base"
DEFAULT_BATCH_SIZE = 32

SPACY_MODEL_NAME = "en_core_web_lg"
TEXT_ENCODER_NAME_OR_PATH = "/home/xiaoyue/ProtoGraphRAG/deberta-v3-large"
EMBEDDING_BATCH_SIZE = 8
REMOVE_DUPLICATE_TOKEN = True
DISCARD_NO_WORD = False

# Set to None to use all HotpotQA documents, or set an integer such as 1000 / 5000.
HOTPOT_NUM_SAMPLES = 500


class ConsoleProgressHandle:
    def __init__(self, initial_text: str):
        self.last_text = ""
        self.update(initial_text)

    def update(self, text: str):
        self.last_text = str(text)
        print(f"\r{self.last_text}", end="", flush=True)

    def close(self):
        if self.last_text:
            print()


def create_progress_handle(initial_text: str):
    handle = display(initial_text, display_id=True)
    if hasattr(handle, "update"):
        return handle
    return ConsoleProgressHandle(initial_text)


def finalize_progress_handle(handle):
    if hasattr(handle, "close"):
        handle.close()
    else:
        print()


def save_pickle(obj, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    try:
        with temp_path.open("wb") as handle:
            pickle.dump(obj, handle)
        temp_path.replace(path)
    finally:
        if temp_path.exists():
            temp_path.unlink()


def load_pickle(path: Path):
    path = Path(path)
    with path.open("rb") as handle:
        return pickle.load(handle)


def load_custom_nlp(model_name: str):
    loaded_nlp = spacy.load(model_name)
    infixes = [pattern for pattern in loaded_nlp.Defaults.infixes if "-" not in pattern]
    infix_re = compile_infix_regex(infixes)

    loaded_nlp.tokenizer = Tokenizer(
        loaded_nlp.vocab,
        prefix_search=loaded_nlp.tokenizer.prefix_search,
        suffix_search=loaded_nlp.tokenizer.suffix_search,
        infix_finditer=infix_re.finditer,
        token_match=loaded_nlp.tokenizer.token_match,
    )
    return loaded_nlp


def load_text_encoder(name_or_path: str, device: str):
    loaded_tokenizer = AutoTokenizer.from_pretrained(
        name_or_path,
        local_files_only=True,
        fix_mistral_regex=True,
        use_fast=True,
    )
    if not getattr(loaded_tokenizer, "is_fast", False):
        raise TypeError(
            "encode_chunk_batch requires a fast tokenizer because it uses return_offset_mapping=True"
        )

    loaded_text_encoder = AutoModel.from_pretrained(
        name_or_path,
        local_files_only=True,
    )
    loaded_text_encoder.to(device)
    loaded_text_encoder.eval()
    return loaded_tokenizer, loaded_text_encoder


def collect_span_embedding_records(token_embeddings, offsets, span_list, kind, document_idx, title):
    records = []
    for term, start_char, end_char in span_list:
        token_indices = get_token_indices_for_phrase(start_char, end_char, offsets)
        if not token_indices:
            continue

        embedding = token_embeddings[token_indices].mean(dim=0).to(torch.float32).numpy()
        records.append(
            {
                "kind": kind,
                "term": term,
                "document_idx": int(document_idx),
                "title": title,
                "span": (int(start_char), int(end_char)),
                "embedding": embedding,
            }
        )
    return records


def build_phrase_token_embedding_index(
    documents,
    nlp,
    text_encoder,
    tokenizer,
    device,
    batch_size=EMBEDDING_BATCH_SIZE,
    remove_duplicate_token=REMOVE_DUPLICATE_TOKEN,
    discard_no_word=DISCARD_NO_WORD,
):
    cleaned_documents = [clean_text(doc["text"]) for doc in documents]

    store = {
        "documents": documents,
        "cleaned_documents": cleaned_documents,
        "index": defaultdict(list),
        "config": {
            "batch_size": batch_size,
            "remove_duplicate_token": remove_duplicate_token,
            "discard_no_word": discard_no_word,
            "encoder": text_encoder.__class__.__name__,
            "tokenizer": tokenizer.__class__.__name__,
            "device": device,
            "text_encoder_name_or_path": TEXT_ENCODER_NAME_OR_PATH,
            "spacy_model_name": SPACY_MODEL_NAME,
        },
    }

    phrase_occurrences = 0
    token_occurrences = 0
    total_documents = len(documents)
    print(f"Total documents to process: {total_documents}")
    progress_handle = create_progress_handle(f"Processed 0/{total_documents} documents")

    try:
        for batch_start in range(0, total_documents, batch_size):
            batch_end = min(batch_start + batch_size, total_documents)
            batch_docs = documents[batch_start:batch_end]
            batch_texts = [cleaned_documents[idx] for idx in range(batch_start, batch_end)]

            batch_spans = [
                extract_important_spans(
                    text,
                    nlp,
                    min_tokens=2,
                    remove_duplicate=remove_duplicate_token,
                    discard_no_word=discard_no_word,
                )
                for text in batch_texts
            ]

            token_embeddings_batch, offsets_batch = encode_chunk_batch(
                batch_texts,
                text_encoder,
                tokenizer,
                device,
            )

            for local_idx, ((phrases, tokens), token_embeddings, offsets) in enumerate(
                zip(batch_spans, token_embeddings_batch, offsets_batch)
            ):
                document_idx = batch_start + local_idx
                title = batch_docs[local_idx]["title"]

                phrase_records = collect_span_embedding_records(
                    token_embeddings,
                    offsets,
                    phrases,
                    kind="phrase",
                    document_idx=document_idx,
                    title=title,
                )
                token_records = collect_span_embedding_records(
                    token_embeddings,
                    offsets,
                    tokens,
                    kind="token",
                    document_idx=document_idx,
                    title=title,
                )

                for record in phrase_records:
                    store["index"][record["term"]].append(record)
                for record in token_records:
                    store["index"][record["term"]].append(record)

                phrase_occurrences += len(phrase_records)
                token_occurrences += len(token_records)

            progress_handle.update(f"Processed {batch_end}/{total_documents} documents")
    finally:
        finalize_progress_handle(progress_handle)

    store["stats"] = {
        "num_documents": len(documents),
        "num_unique_terms": len(store["index"]),
        "num_phrase_occurrences": phrase_occurrences,
        "num_token_occurrences": token_occurrences,
        "num_total_occurrences": phrase_occurrences + token_occurrences,
    }
    return store


def build_hotpot_embedding_store(
    store_path: Path,
    hotpot_file_path: Path | None = None,
    num_samples=None,
):
    if hotpot_file_path is None:
        hotpot_file_path = next(
            (path for path in HOTPOT_FILE_CANDIDATES if path.exists()),
            HOTPOT_FILE_CANDIDATES[0],
        )

    hotpot_file_path = Path(hotpot_file_path)
    if not hotpot_file_path.exists():
        raise FileNotFoundError(
            f"HotpotQA source file not found at {hotpot_file_path}."
        )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Building HotpotQA embedding store from {hotpot_file_path}")
    print(f"Using device: {device}")
    print(f"spaCy model: {SPACY_MODEL_NAME}")
    print(f"text encoder: {TEXT_ENCODER_NAME_OR_PATH}")
    print(f"HOTPOT_NUM_SAMPLES: {num_samples}")

    documents, samples = build_hotpot_retrieval_dataset(str(hotpot_file_path), num_samples=num_samples)
    print(f"documents: {len(documents)}")
    print(f"samples: {len(samples)}")

    nlp = load_custom_nlp(SPACY_MODEL_NAME)
    tokenizer, text_encoder = load_text_encoder(TEXT_ENCODER_NAME_OR_PATH, device)

    store = build_phrase_token_embedding_index(
        documents,
        nlp,
        text_encoder,
        tokenizer,
        device,
        batch_size=EMBEDDING_BATCH_SIZE,
        remove_duplicate_token=REMOVE_DUPLICATE_TOKEN,
        discard_no_word=DISCARD_NO_WORD,
    )
    save_pickle(store, store_path)
    print(f"Saved HotpotQA embedding store to {store_path}")
    print(store.get("stats", {}))
    return store


def load_hotpot_embedding_store(store_path: Path | None = None, build_if_missing: bool = True, num_samples=HOTPOT_NUM_SAMPLES):
    if store_path is None:
        store_path = next(
            (path for path in HOTPOT_EMBEDDING_STORE_CANDIDATES if path.exists()),
            HOTPOT_EMBEDDING_STORE_CANDIDATES[0],
        )

    store_path = Path(store_path)
    if not store_path.exists():
        if not build_if_missing:
            raise FileNotFoundError(
                f"HotpotQA embedding store not found at {store_path}. "
                "Please create hotpot_phrase_token_embedding_index_all.pkl first."
            )
        return build_hotpot_embedding_store(store_path=store_path, num_samples=num_samples)

    with store_path.open("rb") as handle:
        store = pickle.load(handle)

    print(f"Loaded HotpotQA embedding store from {store_path}")
    print(store.get("stats", {}))
    return store


embedding_store = load_hotpot_embedding_store()


Loaded HotpotQA embedding store from /home/xiaoyue/LiteSemRAG/jupyter_notebooks/hotpot_phrase_token_embedding_index_all.pkl
{'num_documents': 4937, 'num_unique_terms': 64422, 'num_phrase_occurrences': 76311, 'num_token_occurrences': 43770, 'num_total_occurrences': 120081}


In [4]:
def lookup_records(store, query_text, kind=None, include_text=True, include_cleaned_text=True):
    normalized_query = normalize_text(query_text.strip())
    records = list(store["index"].get(normalized_query, []))

    if kind is not None:
        records = [record for record in records if record["kind"] == kind]

    if not include_text and not include_cleaned_text:
        return records

    enriched_records = []
    for record in records:
        item = dict(record)
        document_idx = item["document_idx"]
        if include_text:
            item["text"] = store["documents"][document_idx]["text"]
        if include_cleaned_text:
            item["cleaned_text"] = store["cleaned_documents"][document_idx]
        enriched_records.append(item)
    return enriched_records


def extract_sentence_context(cleaned_text, span):
    start_char, end_char = span
    left_boundary = max(
        cleaned_text.rfind(".", 0, start_char),
        cleaned_text.rfind("!", 0, start_char),
        cleaned_text.rfind("?", 0, start_char),
    )
    right_candidates = [
        cleaned_text.find(".", end_char),
        cleaned_text.find("!", end_char),
        cleaned_text.find("?", end_char),
    ]
    right_candidates = [idx for idx in right_candidates if idx != -1]

    context_start = 0 if left_boundary == -1 else left_boundary + 1
    context_end = len(cleaned_text) if not right_candidates else min(right_candidates) + 1
    context_raw = cleaned_text[context_start:context_end]

    if not context_raw.strip():
        context_start = max(0, start_char - 120)
        context_end = min(len(cleaned_text), end_char + 120)
        context_raw = cleaned_text[context_start:context_end]

    left_trim = len(context_raw) - len(context_raw.lstrip())
    context_text = context_raw.strip()

    local_start = start_char - context_start - left_trim
    local_end = end_char - context_start - left_trim
    local_start = max(0, min(local_start, len(context_text)))
    local_end = max(local_start, min(local_end, len(context_text)))

    return {
        "context_text": context_text,
        "local_span": (local_start, local_end),
    }


def build_hotpot_prompt(record, query_text, mark_target=False, left_marker="[TGT]", right_marker="[/TGT]"):
    context_info = extract_sentence_context(record["cleaned_text"], record["span"])
    context_text = context_info["context_text"]
    local_start, local_end = context_info["local_span"]
    prompt_context = context_text

    if mark_target:
        prompt_context = (
            f"{context_text[:local_start]}{left_marker} "
            f"{context_text[local_start:local_end]} "
            f"{right_marker}{context_text[local_end:]}"
        )

    prompt_text = (
        f"Sentence: {prompt_context}\n"
        f"Target word: {query_text.strip()}\n\n"
        f'Question: What does "{query_text.strip()}" mean in this sentence?'
    )

    return {
        "context_text": context_text,
        "matched_text": context_text[local_start:local_end],
        "local_span": (local_start, local_end),
        "prompt_text": prompt_text,
    }


In [5]:
def _stack_record_embeddings(records):
    if not records:
        return np.empty((0, 0), dtype=np.float32)
    return np.stack([np.asarray(record["embedding"], dtype=np.float32) for record in records]).astype(np.float32)


def _l2_normalize_rows(matrix, eps=1e-12):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms < eps] = 1.0
    return matrix / norms


def _l2_normalize_vector(vector, eps=1e-12):
    norm = np.linalg.norm(vector)
    if norm < eps:
        return vector
    return vector / norm


def _compute_cluster_mean_embeddings(embeddings, clusters):
    if not clusters:
        return {}

    normalized_embeddings = _l2_normalize_rows(embeddings)
    cluster_centers = {}
    for label, indices in clusters.items():
        center = normalized_embeddings[indices].mean(axis=0)
        cluster_centers[int(label)] = _l2_normalize_vector(center.astype(np.float32))
    return cluster_centers


def _relabel_clusters_sequential(clusters, cluster_centers=None):
    if not clusters:
        return {}, {}, {}

    ordered_labels = sorted(int(label) for label in clusters.keys())
    relabel_map = {old_label: new_label for new_label, old_label in enumerate(ordered_labels)}

    relabeled_clusters = {
        relabel_map[int(label)]: sorted(int(index) for index in indices)
        for label, indices in clusters.items()
    }
    relabeled_centers = {}
    if cluster_centers is not None:
        relabeled_centers = {
            relabel_map[int(label)]: center
            for label, center in cluster_centers.items()
        }
    return relabeled_clusters, relabel_map, relabeled_centers


def resolve_noise_assignments(records, raw_clusters):
    record_count = len(records)
    if record_count == 0:
        return {
            "clusters": {},
            "cluster_centers": {},
            "noise_indices": [],
            "noise_reassignments": {},
            "all_noise_to_single_cluster": False,
        }

    embeddings = _stack_record_embeddings(records)
    raw_clusters = {
        int(label): [int(index) for index in indices]
        for label, indices in raw_clusters.items()
    }
    noise_indices = sorted(raw_clusters.get(-1, []))
    non_noise_clusters = {
        int(label): sorted(indices)
        for label, indices in raw_clusters.items()
        if int(label) != -1 and indices
    }

    if not raw_clusters:
        single_cluster = {0: list(range(record_count))}
        return {
            "clusters": single_cluster,
            "cluster_centers": _compute_cluster_mean_embeddings(embeddings, single_cluster),
            "noise_indices": [],
            "noise_reassignments": {},
            "all_noise_to_single_cluster": False,
        }

    if not noise_indices:
        cluster_centers = _compute_cluster_mean_embeddings(embeddings, non_noise_clusters)
        final_clusters, relabel_map, final_centers = _relabel_clusters_sequential(
            non_noise_clusters,
            cluster_centers=cluster_centers,
        )
        return {
            "clusters": final_clusters,
            "cluster_centers": final_centers,
            "noise_indices": [],
            "noise_reassignments": {},
            "all_noise_to_single_cluster": False,
        }

    if not non_noise_clusters:
        single_cluster = {0: list(range(record_count))}
        return {
            "clusters": single_cluster,
            "cluster_centers": _compute_cluster_mean_embeddings(embeddings, single_cluster),
            "noise_indices": noise_indices,
            "noise_reassignments": {index: 0 for index in noise_indices},
            "all_noise_to_single_cluster": True,
        }

    normalized_embeddings = _l2_normalize_rows(embeddings)
    cluster_centers = _compute_cluster_mean_embeddings(embeddings, non_noise_clusters)
    resolved_clusters = {
        int(label): list(indices)
        for label, indices in non_noise_clusters.items()
    }
    noise_reassignments = {}

    for record_index in noise_indices:
        similarity_by_cluster = {
            int(label): float(normalized_embeddings[record_index] @ center)
            for label, center in cluster_centers.items()
        }
        best_cluster_label = max(
            similarity_by_cluster,
            key=lambda label: (similarity_by_cluster[label], -label),
        )
        resolved_clusters[best_cluster_label].append(int(record_index))
        noise_reassignments[int(record_index)] = int(best_cluster_label)

    for indices in resolved_clusters.values():
        indices.sort()

    final_clusters, relabel_map, final_centers = _relabel_clusters_sequential(
        resolved_clusters,
        cluster_centers=cluster_centers,
    )
    final_noise_reassignments = {
        int(record_index): int(relabel_map[int(cluster_label)])
        for record_index, cluster_label in noise_reassignments.items()
    }

    return {
        "clusters": final_clusters,
        "cluster_centers": final_centers,
        "noise_indices": noise_indices,
        "noise_reassignments": final_noise_reassignments,
        "all_noise_to_single_cluster": False,
    }


def cluster_single_span_records(records, merge_chunks=False):
    if not records:
        raise ValueError("records must be non-empty.")

    if len(records) == 1:
        embeddings = _stack_record_embeddings(records)
        single_cluster = {0: [0]}
        return {
            "record_count": 1,
            "min_cluster_size": 1,
            "raw_n_clusters": 1,
            "raw_clusters": {0: [0]},
            "clusters": single_cluster,
            "cluster_centers": _compute_cluster_mean_embeddings(embeddings, single_cluster),
            "noise_indices": [],
            "noise_reassignments": {},
            "all_noise_to_single_cluster": False,
        }

    min_cluster_size = max(2, (len(records) + 9) // 10)
    embeds_list = [(torch.as_tensor(record["embedding"]), None) for record in records]
    raw_n_clusters, raw_clusters, _ = hdbscan_cluster(
        embeds_list,
        min_cluster_size=min_cluster_size,
        merge_chunks=merge_chunks,
    )
    raw_clusters = {
        int(label): sorted(int(index) for index in indices)
        for label, indices in raw_clusters.items()
    }
    resolved = resolve_noise_assignments(records, raw_clusters)
    return {
        "record_count": len(records),
        "min_cluster_size": min_cluster_size,
        "raw_n_clusters": int(raw_n_clusters),
        "raw_clusters": raw_clusters,
        "clusters": resolved["clusters"],
        "cluster_centers": resolved["cluster_centers"],
        "noise_indices": resolved["noise_indices"],
        "noise_reassignments": resolved["noise_reassignments"],
        "all_noise_to_single_cluster": resolved["all_noise_to_single_cluster"],
    }


def build_compact_cluster_result(span_text, cluster_result, kind=None):
    raw_cluster_sizes = {
        int(label): len(indices)
        for label, indices in cluster_result["raw_clusters"].items()
    }
    final_cluster_sizes = {
        int(label): len(indices)
        for label, indices in cluster_result["clusters"].items()
    }
    return {
        "query_text": span_text,
        "kind": kind,
        "record_count": int(cluster_result["record_count"]),
        "min_cluster_size": int(cluster_result["min_cluster_size"]),
        "raw_n_clusters": int(cluster_result["raw_n_clusters"]),
        "raw_cluster_sizes": raw_cluster_sizes,
        "raw_noise_count": len(cluster_result["noise_indices"]),
        "noise_indices": list(cluster_result["noise_indices"]),
        "noise_reassignments": dict(cluster_result["noise_reassignments"]),
        "all_noise_to_single_cluster": bool(cluster_result["all_noise_to_single_cluster"]),
        "clusters": {
            int(label): list(indices)
            for label, indices in cluster_result["clusters"].items()
        },
        "cluster_sizes": final_cluster_sizes,
        "final_cluster_count": len(cluster_result["clusters"]),
    }


def cluster_all_spans_in_store(
    store,
    kind=None,
    merge_chunks=False,
    output_path=DEFAULT_CLUSTER_RESULTS_PATH,
    print_every=5000,
):
    terms = sorted(store["index"].keys())
    results = {}
    multi_cluster_spans = 0

    for span_index, span_text in enumerate(terms, start=1):
        records = lookup_records(
            store,
            span_text,
            kind=kind,
            include_text=False,
            include_cleaned_text=False,
        )
        if not records:
            continue

        cluster_result = cluster_single_span_records(records, merge_chunks=merge_chunks)
        compact_result = build_compact_cluster_result(
            span_text,
            cluster_result,
            kind=kind,
        )
        results[span_text] = compact_result

        if compact_result["final_cluster_count"] > 1:
            multi_cluster_spans += 1

        if span_index % print_every == 0 or span_index == len(terms):
            print(
                f"Clustered {span_index}/{len(terms)} spans | "
                f"multi_cluster_spans={multi_cluster_spans}"
            )

    payload = {
        "config": {
            "kind": kind,
            "merge_chunks": merge_chunks,
            "store_path_candidates": [str(path) for path in HOTPOT_EMBEDDING_STORE_CANDIDATES],
        },
        "summary": {
            "num_total_spans": len(terms),
            "num_clustered_spans": len(results),
            "num_multi_cluster_spans": multi_cluster_spans,
        },
        "results": results,
    }
    save_pickle(payload, output_path)
    print(f"Saved cluster results to {output_path}")
    print(payload["summary"])
    return payload


In [6]:
def _prepare_candidate_bank(
    span_text,
    use_detailed_description=False,
    exact_match_text=True,
    limit=5,
):
    candidates_df, definition_column = load_wikidata_definition_candidates(
        span_text,
        use_detailed_description=use_detailed_description,
        exact_match_text=exact_match_text,
        limit=limit,
    )
    candidate_bank = build_wikidata_candidate_bank(
        candidates_df,
        definition_column=definition_column,
    )
    return candidates_df, definition_column, candidate_bank


def predict_labels_for_records(
    records,
    span_text,
    model,
    candidate_bank,
    batch_size=DEFAULT_BATCH_SIZE,
    mark_target=False,
):
    if not records:
        return []
    if not candidate_bank:
        raise ValueError("candidate_bank must be non-empty.")

    prompt_infos = [
        build_hotpot_prompt(record, span_text, mark_target=mark_target)
        for record in records
    ]
    pairs = [
        (prompt_info["prompt_text"], candidate["hypothesis"])
        for prompt_info in prompt_infos
        for candidate in candidate_bank
    ]
    raw_scores = model.predict(
        pairs,
        batch_size=min(batch_size, len(pairs)),
        show_progress_bar=False,
    )
    score_matrix = extract_cross_encoder_scores(raw_scores, model).reshape(
        len(records),
        len(candidate_bank),
    )

    predictions = []
    for record_index, (record, prompt_info, row_scores) in enumerate(
        zip(records, prompt_infos, score_matrix)
    ):
        best_candidate_index = int(np.argmax(row_scores))
        top_candidate = candidate_bank[best_candidate_index]
        predictions.append(
            {
                "record_index": int(record_index),
                "title": record["title"],
                "kind": record["kind"],
                "document_idx": int(record["document_idx"]),
                "span": tuple(record["span"]),
                "matched_text": prompt_info["matched_text"],
                "context_text": prompt_info["context_text"],
                "predicted_entity_id": top_candidate["entity_id"],
                "predicted_label": top_candidate["label"],
                "predicted_description": top_candidate["description"],
                "predicted_definition": top_candidate["definition"],
                "prediction_score": float(row_scores[best_candidate_index]),
            }
        )

    return predictions


def summarize_cluster_predictions(cluster_predictions):
    if not cluster_predictions:
        return {
            "assigned_entity_id": None,
            "assigned_label": None,
            "assigned_description": None,
            "assigned_definition": None,
            "label_distribution": [],
        }

    description_counter = Counter(
        item["predicted_description"]
        for item in cluster_predictions
    )
    assigned_description = description_counter.most_common(1)[0][0]

    assigned_candidates = [
        item
        for item in cluster_predictions
        if item["predicted_description"] == assigned_description
    ]
    assigned_candidate_counter = Counter(
        (
            item["predicted_entity_id"],
            item["predicted_label"],
            item["predicted_description"],
            item["predicted_definition"],
        )
        for item in assigned_candidates
    )
    assigned_entity_id, assigned_label, _, assigned_definition = assigned_candidate_counter.most_common(1)[0][0]

    full_label_counter = Counter(
        (
            item["predicted_entity_id"],
            item["predicted_label"],
            item["predicted_description"],
            item["predicted_definition"],
        )
        for item in cluster_predictions
    )

    total_predictions = len(cluster_predictions)
    label_distribution = []
    for (
        entity_id,
        label,
        description,
        definition,
    ), count in full_label_counter.most_common():
        label_distribution.append(
            {
                "predicted_entity_id": entity_id,
                "predicted_label": label,
                "predicted_description": description,
                "predicted_definition": definition,
                "count": int(count),
                "ratio": float(count / total_predictions),
            }
        )

    return {
        "assigned_entity_id": assigned_entity_id,
        "assigned_label": assigned_label,
        "assigned_description": assigned_description,
        "assigned_definition": assigned_definition,
        "label_distribution": label_distribution,
    }


def print_span_cluster_label_breakdown(span_result):
    span_text = span_result["query_text"]
    print("=" * 120)
    print(
        f"Span: {span_text!r} | record_count={span_result['record_count']} | "
        f"cluster_count={span_result['cluster_count']}"
    )

    if span_result.get("error"):
        print(f"Label assignment failed: {span_result['error']}")
        return

    if span_result["merge_groups"]:
        print("Duplicate-description merge groups:")
        for item in span_result["merge_groups"]:
            print(
                f"  description={item['description']!r} | cluster_labels={item['cluster_labels']}"
            )
    else:
        print("Duplicate-description merge groups: none")

    for cluster_summary in span_result["cluster_summaries"]:
        print(
            f"  cluster {cluster_summary['cluster_label']} | size={cluster_summary['cluster_size']} | "
            f"assigned_label={cluster_summary['assigned_label']!r} | "
            f"assigned_description={cluster_summary['assigned_description']!r}"
        )
        for row in cluster_summary["label_distribution"]:
            print(
                "    "
                f"{row['predicted_label']!r} | {row['predicted_description']!r}: "
                f"{row['count']} ({row['ratio']:.2%})"
            )


def label_multi_cluster_spans(
    store,
    cluster_payload,
    model_name=DEFAULT_MODEL_NAME,
    batch_size=DEFAULT_BATCH_SIZE,
    use_detailed_description=False,
    exact_match_text=True,
    mark_target=False,
    output_path=DEFAULT_LABEL_RESULTS_PATH,
    print_output=True,
    print_every=50,
):
    cluster_results = cluster_payload["results"] if "results" in cluster_payload else cluster_payload
    multi_cluster_items = [
        (span_text, result)
        for span_text, result in cluster_results.items()
        if int(result.get("final_cluster_count", 0)) > 1
    ]

    print(f"Found {len(multi_cluster_items)} spans with more than one cluster.")
    model = CrossEncoder(model_name)
    results = {}

    for span_index, (span_text, cluster_result) in enumerate(multi_cluster_items, start=1):
        records = lookup_records(
            store,
            span_text,
            kind=cluster_result.get("kind"),
            include_text=True,
            include_cleaned_text=True,
        )

        span_result = {
            "query_text": span_text,
            "kind": cluster_result.get("kind"),
            "record_count": len(records),
            "cluster_count": int(cluster_result["final_cluster_count"]),
            "cluster_summaries": [],
            "record_predictions": [],
            "candidate_rows": [],
            "merge_groups": [],
            "error": None,
        }

        try:
            candidates_df, definition_column, candidate_bank = _prepare_candidate_bank(
                span_text,
                use_detailed_description=use_detailed_description,
                exact_match_text=exact_match_text,
                limit=5,
            )
            span_result["candidate_rows"] = candidates_df[
                ["id", "label", "description", definition_column]
            ].to_dict("records")

            predictions = predict_labels_for_records(
                records,
                span_text,
                model,
                candidate_bank,
                batch_size=batch_size,
                mark_target=mark_target,
            )
            span_result["record_predictions"] = predictions
            predictions_by_index = {
                int(item["record_index"]): item
                for item in predictions
            }

            assigned_description_groups = defaultdict(list)
            for cluster_label, indices in sorted(cluster_result["clusters"].items()):
                cluster_predictions = [
                    predictions_by_index[int(record_index)]
                    for record_index in indices
                ]
                cluster_summary = summarize_cluster_predictions(cluster_predictions)
                cluster_summary["cluster_label"] = int(cluster_label)
                cluster_summary["cluster_size"] = len(indices)
                cluster_summary["record_indices"] = list(indices)
                span_result["cluster_summaries"].append(cluster_summary)
                if cluster_summary["assigned_description"] is not None:
                    assigned_description_groups[cluster_summary["assigned_description"]].append(
                        int(cluster_label)
                    )

            span_result["merge_groups"] = [
                {
                    "description": description,
                    "cluster_labels": labels,
                }
                for description, labels in assigned_description_groups.items()
                if len(labels) > 1
            ]
        except Exception as exc:
            span_result["error"] = str(exc)

        results[span_text] = span_result

        if print_output:
            print_span_cluster_label_breakdown(span_result)

        if span_index % print_every == 0 or span_index == len(multi_cluster_items):
            print(
                f"Labeled {span_index}/{len(multi_cluster_items)} multi-cluster spans"
            )

    payload = {
        "config": {
            "model_name": model_name,
            "batch_size": batch_size,
            "use_detailed_description": use_detailed_description,
            "exact_match_text": exact_match_text,
            "mark_target": mark_target,
        },
        "summary": {
            "num_multi_cluster_spans": len(multi_cluster_items),
            "num_completed_spans": len(results),
            "num_failed_spans": sum(
                1
                for item in results.values()
                if item.get("error")
            ),
        },
        "results": results,
    }
    save_pickle(payload, output_path)
    print(f"Saved label results to {output_path}")
    print(payload["summary"])
    return payload


## Example Usage

Run the full clustering pass first, then run exhaustive label assignment for spans with more than one cluster.


In [7]:
#Full clustering over every span in the HotpotQA embedding store.
cluster_payload = cluster_all_spans_in_store(
    embedding_store,
    kind=None,
    merge_chunks=False,
    output_path=DEFAULT_CLUSTER_RESULTS_PATH,
    print_every=5000,
)


#Exhaustive label assignment for all multi-cluster spans.
label_payload = label_multi_cluster_spans(
    embedding_store,
    cluster_payload=cluster_payload,
    model_name=DEFAULT_MODEL_NAME,
    batch_size=DEFAULT_BATCH_SIZE,
    use_detailed_description=False,
    exact_match_text=True,
    mark_target=False,
    output_path=DEFAULT_LABEL_RESULTS_PATH,
    print_output=True,
    print_every=25,
)


Clustered 5000/64422 spans | multi_cluster_spans=14
Clustered 10000/64422 spans | multi_cluster_spans=77
Clustered 15000/64422 spans | multi_cluster_spans=170
Clustered 20000/64422 spans | multi_cluster_spans=262
Clustered 25000/64422 spans | multi_cluster_spans=323
Clustered 30000/64422 spans | multi_cluster_spans=396
Clustered 35000/64422 spans | multi_cluster_spans=468
Clustered 40000/64422 spans | multi_cluster_spans=535
Clustered 45000/64422 spans | multi_cluster_spans=612
Clustered 50000/64422 spans | multi_cluster_spans=704
Clustered 55000/64422 spans | multi_cluster_spans=784
Clustered 60000/64422 spans | multi_cluster_spans=876
Clustered 64422/64422 spans | multi_cluster_spans=950
Saved cluster results to /home/xiaoyue/LiteSemRAG/jupyter_notebooks/hotpot_hdbscan_cluster_results.pkl
{'num_total_spans': 64422, 'num_clustered_spans': 64421, 'num_multi_cluster_spans': 950}
Found 950 spans with more than one cluster.


/tmp/ipykernel_1855492/3498989431.py:236: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  ].to_dict("records")


Span: '10,000 metres' | record_count=9 | cluster_count=2
Duplicate-description merge groups:
  description='common long distance running event' | cluster_labels=[0, 1]
  cluster 0 | size=5 | assigned_label='10,000 metres' | assigned_description='common long distance running event'
    '10,000 metres' | 'common long distance running event': 5 (100.00%)
  cluster 1 | size=4 | assigned_label='10,000 metres' | assigned_description='common long distance running event'
    '10,000 metres' | 'common long distance running event': 4 (100.00%)


/tmp/ipykernel_1855492/3498989431.py:236: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  ].to_dict("records")


Span: '13th century' | record_count=17 | cluster_count=2
Duplicate-description merge groups:
  description='century' | cluster_labels=[0, 1]
  cluster 0 | size=6 | assigned_label='13th century' | assigned_description='century'
    '13th century' | 'century': 6 (100.00%)
  cluster 1 | size=11 | assigned_label='13th century' | assigned_description='century'
    '13th century' | 'century': 11 (100.00%)


...

Span: 'mission' | record_count=16 | cluster_count=3
Duplicate-description merge groups:
  description='organized effort for the propagation of the Christian faith' | cluster_labels=[0, 2]
  cluster 0 | size=5 | assigned_label='Christian mission' | assigned_description='organized effort for the propagation of the Christian faith'
    'Christian mission' | 'organized effort for the propagation of the Christian faith': 3 (60.00%)
    'Mission' | 'city in Hidalgo County, Texas, United States': 2 (40.00%)
  cluster 1 | size=7 | assigned_label='filial church' | assigned_description='church building that is not the main church of a parish'
    'filial church' | 'church building that is not the main church of a parish': 2 (28.57%)
    'Mission' | 'city in Hidalgo County, Texas, United States': 2 (28.57%)
    'military operation' | 'coordinated military actions of a state or a non-state actor': 2 (28.57%)
    'Christian mission' | 'organized effort for the propagation of the Christian faith': 1

/tmp/ipykernel_1855492/3498989431.py:236: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  ].to_dict("records")


Span: 'mississippi' | record_count=13 | cluster_count=2
Duplicate-description merge groups:
  description='state of the United States of America' | cluster_labels=[0, 1]
  cluster 0 | size=4 | assigned_label='Mississippi' | assigned_description='state of the United States of America'
    'Mississippi' | 'state of the United States of America': 4 (100.00%)
  cluster 1 | size=9 | assigned_label='Mississippi' | assigned_description='state of the United States of America'
    'Mississippi' | 'state of the United States of America': 8 (88.89%)
    'Mississippi' | 'vocal track by Vikingarna; 1976 studio recording': 1 (11.11%)
Span: 'missouri' | record_count=43 | cluster_count=2
Duplicate-description merge groups:
  description='state of the United States of America' | cluster_labels=[0, 1]
  cluster 0 | size=18 | assigned_label='Missouri' | assigned_description='state of the United States of America'
    'Missouri' | 'state of the United States of America': 18 (100.00%)
  cluster 1 | size=25

/tmp/ipykernel_1855492/3498989431.py:236: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  ].to_dict("records")


KeyboardInterrupt: 